# Graph based analysis of interactions in Litigi community


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

from subreddit_lens import load_config

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
DATA_DIR = config.data_dir
OUTPUT_DIR = config.output_dir
OUTPUT_DIR.mkdir(exist_ok=True)
from subreddit_lens import (
    create_nx_graph,
    extract_interaction_graph,
    get_parent_author,
    hindex,
    load_comments,
    load_submissions,
    preprocess,
    symmetrize_graph,
)

## Load and preprocess data

In [ ]:
filename = DATA_DIR / "litigi_comments.parquet"
litigi = load_comments(filename)
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

# Submissions give the author of each post, so that replies to posts count
# as interactions with their author.
submissions_file = DATA_DIR / "litigi_submissions.parquet"
submissions = load_submissions(submissions_file) if submissions_file.exists() else None

# extract the author of the parent comment or post (if applicable)
litigi = get_parent_author(litigi, submissions)

litigi.head()

## Extract subsample

In [ ]:
litigi_subsample=litigi[:50000][['author','body_preprocessed','id', 'is_submitter', 'link_id', 'parent_id','created_dt','parent_author']]
litigi_subsample

# how many replies to top level comments?

In [ ]:
first_level_comments=litigi[litigi['parent_id'].str.startswith('t3')][['author','id']].reset_index(drop=True)
first_level_comments

In [ ]:
first_level_comments_id=list(first_level_comments['id'])

In [ ]:
top_users_by_comments=litigi.groupby('author').size().sort_values(ascending=False).reset_index().rename(columns={0:'comments'})
top_users_by_comments=top_users_by_comments[top_users_by_comments['author'] != '[deleted]']
top_users_by_comments

In [ ]:
# Replies to comments only: comment and submission IDs can coincide.
replies_to_comments=litigi[litigi['parent_id'].str.startswith('t1_')]
number_of_replies=replies_to_comments.groupby(['parent_author','parent_id']).size().reset_index().rename(columns={0:'num_replies','parent_id':'id','parent_author':'author'})
number_of_replies['id']=number_of_replies['id'].str.split('_').str[1]
number_of_replies=pd.concat([number_of_replies,first_level_comments]).drop_duplicates(['author','id'])
number_of_replies['is_first_level']=number_of_replies['id'].isin(first_level_comments_id)
number_of_replies=number_of_replies.fillna(0)
number_of_replies

In [ ]:
avg_replies=number_of_replies[number_of_replies['is_first_level']].reset_index(drop=True).groupby('author')['num_replies'].mean().sort_values(ascending=False).reset_index()
avg_replies

In [ ]:
avg_replies_topx=avg_replies[avg_replies['author'].isin(top_users_by_comments['author'][:250])].reset_index(drop=True)
avg_replies_topx

## Comment chains

In [ ]:
C_Graph= create_nx_graph(litigi)
print(C_Graph)

In [ ]:
#extract the longest comment chain
S = C_Graph.subgraph(max(nx.connected_components(C_Graph.to_undirected()), key=len)).copy()

In [ ]:

S=nx.dag_longest_path(C_Graph)

conversation=litigi[litigi['id'].isin(list(S))]
longest_conv=conversation[['author','id','parent_id','body','created_dt']]
longest_conv

## Extract user interactions

In [ ]:
G=extract_interaction_graph(litigi)
print(G)
pagerank = nx.pagerank(G.reverse())
# Top 20 users by PageRank
sorted(pagerank.items(), key=lambda item: item[1], reverse=True)[:20]

In [ ]:
hindex(G,'innocent_butungu')

In [ ]:
G.in_degree(['RedAliena'],weight='weight') # G.in_degree(['user']) should be the sum of the comment recieved by the user

In [ ]:
G_sym=symmetrize_graph(G)


In [ ]:
edge_weights = nx.get_edge_attributes(G_sym, 'weight')
log_w = {k: np.log(v) for k, v in edge_weights.items()}


## Export and visualize network

In [ ]:
from pyvis.network import Network

nx.set_edge_attributes(G_sym,log_w,'weight')

options = {
  "physics": {
    "enabled": False,
    "barnesHut": {
      "centralGravity": 1.6,
      "springLength": 140,
      "springConstant": 0.1
    },
    "minVelocity": 0.75,
    "timestep": 0.11
  }
}
# Convert NetworkX graph to Pyvis
# cdn_resources='in_line' embeds the JavaScript in the HTML file; the default
# ('local') writes a lib/ folder to the working directory.
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=True, cdn_resources="in_line")
net.show_buttons(filter_=['physics'])
net.from_nx(G_sym)
net.toggle_physics(False)
# save_graph() writes with the platform's default encoding, which fails on
# Windows for the inlined JavaScript.
(OUTPUT_DIR / "litigi.html").write_text(net.generate_html(), encoding="utf-8")

## Sources

https://towardsdatascience.com/large-graph-visualization-tools-and-approaches-2b8758a1cd59?gi=130d1c56065b

For a chord diagram look into: https://plotly.com/python/v3/chord-diagram/